In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://enhantica.github.io/edi-download/pypi/simple/


# Single-crystal fit: Tb₂Ti₂O₇, HEiDi

Fit the pyrochlore Tb₂Ti₂O₇ to constant-wavelength neutron single-crystal data from HEiDi at
MLZ, with anisotropic displacements and Becker-Coppens extinction.

The project is the catalogue case `sc-neut-cwl_tb2ti2o7-heidi`, the one the command line and the
app's Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">sc</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('sc-neut-cwl_tb2ti2o7-heidi', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['tb2ti2o7']
experiment = project.experiments['heidi']

Data 'sc-neut-cwl_tb2ti2o7-heidi' is at data/sc-neut-cwl_tb2ti2o7-heidi-38a48062af12/sc-neut-cwl_tb2ti2o7-heidi


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('heidi')

View(html=<99,715 characters>, height=480)

## Perform the analysis

One new group per fit: the scale, O1's position, the displacement tensors, then the extinction.
diffraction-lib also fits occupancies and isotropic displacements, which this case keeps fixed.

### Scale

The scale brings the calculated intensities to the measured ones.

In [6]:
experiment.linked_structures['tb2ti2o7'].scale.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,heidi,linked_structure,tb2ti2o7,scale,4,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 1 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        22.22        75.58
      1      0.001      19.62        58.90   22.1% ↓
      2      0.002      19.62        58.90   
  done · converged · 2 iters · reduced χ² 58.90 · wR(F²) 19.62 % · 0.00 s


View(html=<77,544 characters>, height=376)

In [9]:
project.display.pattern('heidi')

View(html=<100,087 characters>, height=480)

### O1's position

The one coordinate the symmetry leaves free.

In [10]:
structure.atom_sites['O1'].fract_x.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 2 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        19.62        59.17
      1      0.001      14.74        33.42   43.5% ↓
      2      0.002      14.74        33.42   
  done · converged · 2 iters · reduced χ² 33.42 · wR(F²) 14.74 % · 0.00 s


View(html=<78,720 characters>, height=376)

In [12]:
project.display.pattern('heidi')

View(html=<100,164 characters>, height=480)

### Anisotropic displacements

In [13]:
for tensor in structure.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 10 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        14.74        34.69
      1      0.001       6.20         6.13   82.3% ↓
      2      0.003       6.09         5.93   3.3% ↓
      3      0.005       6.09         5.93   
  done · converged · 3 iters · reduced χ² 5.93 · wR(F²) 6.09 % · 0.01 s


View(html=<87,570 characters>, height=376)

In [15]:
project.display.pattern('heidi')

View(html=<100,521 characters>, height=480)

### Extinction

The domain radius and the mosaic spread.

In [16]:
experiment.extinction.domain_radius.free = True
experiment.extinction.mosaic_spread.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 12 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.09         5.98
      1      0.001       3.86         2.40   59.9% ↓
      2      0.003       3.18         1.63   32.1% ↓
      3      0.004       2.97         1.42   13.1% ↓
      4      0.006       2.92         1.38   2.9% ↓
      5      0.008       2.91         1.37   0.7% ↓
      6      0.009       2.88         1.34   2.0% ↓
      7      0.011       2.88         1.33   0.4% ↓
      8      0.012       2.88         1.33   
      9      0.014       2.88         1.33   
     10      0.016       2.88         1.33   
  done · converged · 10 iters · reduced χ² 1.33 · wR(F²) 2.88 % · 0.02 s


View(html=<89,817 characters>, height=376)

In [18]:
project.display.pattern('heidi')

View(html=<100,548 characters>, height=480)

In [19]:
project.display.fit.correlations()

,1,2,3,4,5
1 tb2ti2o7.atom_site_aniso.Tb.adp_11,,,,,
2 tb2ti2o7.atom_site_aniso.O1.adp_22,0.52,,,,
3 heidi.linked_structure.tb2ti2o7.scale,0.69,0.65,,,
4 heidi.extinction.domain_radius,0.18,0.19,0.17,,
5 heidi.extinction.mosaic_spread,0.19,0.14,0.47,-0.67,


In [20]:
project.display.structure('tb2ti2o7')

View(html=<896,879 characters>, height=660)

## Save the project

In [21]:
project.save_as('sc-neut-cwl_tb2ti2o7-heidi')
project.report.save_html('sc-neut-cwl_tb2ti2o7-heidi.html')

Saving project to 'sc-neut-cwl_tb2ti2o7-heidi'
Saving report to 'sc-neut-cwl_tb2ti2o7-heidi.html'
